# SHAP Analysis — Deep Dive into Feature Importance

Use SHAP to understand which features drive AQI predictions and how
they interact. This notebook complements the dashboard's Feature Importance page.

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import shap

from src.config import settings
from src.training_pipeline.loader import load_feature_view, create_labels, split_data, preprocess
from src.training_pipeline.models import build_random_forest
from src.hopsworks_setup import get_feature_store

print('SHAP version:', shap.__version__)

## 1. Load Data and Train Model

In [ ]:
fs = get_feature_store()
df, _ = load_feature_view(fs)
df = create_labels(df)
X_train, X_test, y_train, y_test = split_data(df)
X_train_s, X_test_s, scaler = preprocess(X_train, X_test)

feature_names = list(X_train.columns)

rf = build_random_forest(n_estimators=100, max_depth=15)  # Smaller for SHAP speed
rf.fit(X_train_s, y_train.values[:, 0])  # Predict day+1 only for simplicity
print(f'Trained RF on {X_train_s.shape[0]} samples, {X_train_s.shape[1]} features')

## 2. Compute SHAP Values

In [ ]:
# Use TreeExplainer for fast SHAP on tree-based models
explainer = shap.TreeExplainer(rf)

# Compute SHAP on a subset of test data (full SHAP can be slow)
X_sample = X_test_s[:200]
shap_values = explainer.shap_values(X_sample)

print(f'SHAP values shape: {shap_values.shape}')

## 3. SHAP Summary (Beeswarm)

In [ ]:
shap.summary_plot(shap_values, X_sample, feature_names=feature_names,
                  max_display=15, show=False)
# Note: the SHAP summary_plot is static matplotlib —
# for interactive Plotly versions see src/dashboard/components/charts.py

## 4. Feature Importance (Bar)

In [ ]:
shap.summary_plot(shap_values, X_sample, feature_names=feature_names,
                  plot_type='bar', max_display=15, show=False)

## 5. Per-City Feature Importance

Do different cities have different important features?

In [ ]:
for city in df['city'].unique()[:3]:
    try:
        city_mask = X_test['city'] == city  # or use index
        city_sample = X_test_s[:200]
        city_shap = explainer.shap_values(city_sample)
        
        mean_abs = np.abs(city_shap).mean(axis=0)
        top_idx = np.argsort(mean_abs)[-5:]
        
        print(f'\n{city} — Top 5 Features:')
        for idx in reversed(top_idx):
            print(f'  {feature_names[idx]:<30s}: {mean_abs[idx]:.4f}')
    except Exception as e:
        print(f'{city}: {e}')

## 6. Waterfall Plot for a Single Prediction

In [ ]:
shap.waterfall_plot(
    shap.Explanation(
        values=shap_values[0],
        base_values=explainer.expected_value,
        data=X_sample[0],
        feature_names=feature_names,
    ),
    max_display=10,
    show=False,
)
print(f'Base (expected) value: {explainer.expected_value:.1f} AQI')
print(f'Predicted value: {rf.predict(X_sample[:1])[0]:.1f} AQI')

## 7. Key Findings

- Document which features consistently appear as most important
- Note any surprising feature interactions
- Consider feature pruning for less important features to speed up training
- Save SHAP values for the dashboard to load (Model Registry)